# Gemma 4 finetuning

In [ ]:
# Install/upgrade Unsloth and common training dependencies for this notebook
%pip install -U pip setuptools wheel

# Pin CUDA-enabled PyTorch to a version compatible with current Unsloth
%pip install --upgrade --force-reinstall --no-cache-dir --index-url https://download.pytorch.org/whl/cu128 torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0

%pip install -U unsloth trl datasets accelerate peft bitsandbytes sentencepiece protobuf
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

# Optional but commonly useful with Unsloth on NVIDIA GPUs
%pip install -U xformers

print("Dependencies installed.")
print("Important: Restart the notebook kernel now, then run cell 2.")

In [ ]:
import os
import subprocess

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
requested_gpus = ["0"]

available_gpus = []
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"],
        text=True,
    )
    available_gpus = [line.split(",", 1)[0].strip() for line in smi_output.splitlines() if line.strip()]
except Exception as e:
    print("nvidia-smi query failed:", e)

if all(gpu in available_gpus for gpu in requested_gpus):
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(requested_gpus)
    print("Using requested GPUs:", os.environ["CUDA_VISIBLE_DEVICES"])
else:
    fallback = ",".join(available_gpus[:2]) if len(available_gpus) >= 2 else ",".join(available_gpus)
    if fallback:
        os.environ["CUDA_VISIBLE_DEVICES"] = fallback
        print(f"Requested GPUs {','.join(requested_gpus)} not found; using available GPUs {fallback} instead.")
    else:
        os.environ.pop("CUDA_VISIBLE_DEVICES", None)
        print("No GPUs were reported by nvidia-smi.")

Using requested GPUs: 0,1


In [2]:
import torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} -> {torch.cuda.get_device_name(i)}")

CUDA_VISIBLE_DEVICES = 0,1
CUDA available: True
Visible GPU count: 2
cuda:0 -> NVIDIA GeForce RTX 5090
cuda:1 -> NVIDIA GeForce RTX 3090


In [3]:
# Per-GPU runtime sanity check: confirms each visible GPU can run CUDA ops
for i in range(torch.cuda.device_count()):
    x = torch.randn(2048, device=f"cuda:{i}")
    y = (x * x).mean().item()
    print(f"cuda:{i} compute check ok, mean={y:.6f}")

cuda:0 compute check ok, mean=1.014838
cuda:1 compute check ok, mean=0.994446


In [4]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset

try:
    from trl import SFTTrainer, SFTConfig
except Exception:
    from trl.trainer.sft_trainer import SFTTrainer
    from trl.trainer.sft_config import SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\li\miniconda3\envs\unsloth\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0402 18:41:15.978000 86552 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Config

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/gemma-4-31B-it"
MAX_SEQ_LENGTH = 8192 # Max Context Length
STAGE1_SHISA_ROWS = 12000
SHISA_REPLAY_ROWS = 3000
STAGE1_MAX_STEPS = 350
STAGE2_MAX_STEPS = 700
GRAD_ACCUM_STEPS = 8
STAGE1_LR = 1.2e-4
STAGE2_LR = 7e-5
BENCHMARK_ROWS = 128
LORA_R = 8
LORA_ALPHA = 16

# Data safety / filtering
ENABLE_NSFW_FILTER = True
APPLY_NSFW_FILTER_TO_STAGE2 = True
APPLY_NSFW_FILTER_TO_BENCHMARK = True
NSFW_FILTER_LEVEL_STAGE2 = "explicit_only"
NSFW_FILTER_LEVEL_BENCHMARK = "minimal"

# Multi-GPU model load behavior (helps 31B fit better).
USE_MULTI_GPU_MODEL_SHARDING = True
MULTI_GPU_DEVICE_MAP = "balanced"
GPU0_RESERVED_GIB = 5
OTHER_GPU_RESERVED_GIB = 3

# ── DIRECTORIES ───────────────────────────────────────────────────────────────
OUTPUT_DIR          = "outputs/stage1"
STAGE2_OUTPUT_DIR   = "outputs/stage2"
FINAL_ADAPTER_DIR   = "outputs/gemma4_31b_lora_final"

# Configure Datasets

In [6]:
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import standardize_data_formats

# Stage 1: broader JA/EN instruction behavior
shisa_stage1_raw = load_dataset(
    "shisa-ai/shisa-v2-sharegpt",
    split=f"train[:{STAGE1_SHISA_ROWS}]",
)
shisa_stage1_raw = standardize_data_formats(shisa_stage1_raw)


# Stage 2: VN specialization + anti-forgetting replay
vntl_stage2_train_raw = load_dataset(
    "lmg-anon/VNTL-v3.1-1k-q",
    split=f"train[:-{BENCHMARK_ROWS}]",
)
vntl_stage2_train_raw = standardize_data_formats(vntl_stage2_train_raw)

vntl_benchmark_raw = load_dataset(
    "lmg-anon/VNTL-v3.1-1k-q",
    split=f"train[-{BENCHMARK_ROWS}:]",
)
vntl_benchmark_raw = standardize_data_formats(vntl_benchmark_raw)

shisa_replay_raw = load_dataset(
    "shisa-ai/shisa-v2-sharegpt",
    split=f"train[{STAGE1_SHISA_ROWS}:{STAGE1_SHISA_ROWS + SHISA_REPLAY_ROWS}]",
)
shisa_replay_raw = standardize_data_formats(shisa_replay_raw)


stage2_raw = concatenate_datasets([vntl_stage2_train_raw, shisa_replay_raw]).shuffle(seed=3407)
benchmark_raw = vntl_benchmark_raw

# Keep `dataset` for your existing preview cells (Stage 1)
dataset = shisa_stage1_raw

print(
    f"Stage1 rows: {len(dataset)} | "
    f"Stage2 rows: {len(stage2_raw)} (VNTL train + replay) | "
    f"Benchmark rows: {len(benchmark_raw)}"
)

print(
    f"Stage1 Shisa: {len(shisa_stage1_raw)} | "
    f"Stage2 VNTL train: {len(vntl_stage2_train_raw)} | "
    f"Stage2 Shisa replay: {len(shisa_replay_raw)} | "
    f"Held-out benchmark rows: {len(vntl_benchmark_raw)}"
)

Stage1 rows: 12000 | Stage2 rows: 13017 (VNTL train + replay) | Benchmark rows: 128
Stage1 Shisa: 12000 | Stage2 VNTL train: 10017 | Stage2 Shisa replay: 3000 | Held-out benchmark rows: 128


### Dataset Postprocessing

In [7]:
from transformers import AutoTokenizer
import re

if "tokenizer" not in globals():
    # Load tokenizer only for dataset rendering; model load happens later in training section.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer = get_chat_template(
        tokenizer,
        chat_template="gemma-4",
    )
    print("Tokenizer initialized for Gemma-4 chat formatting.")

def _is_non_empty_text(x):
    return isinstance(x, str) and len(x.strip()) > 0

def _get_row_value(batch, key, idx):
    values = batch.get(key, None)
    if isinstance(values, list) and idx < len(values):
        return values[idx]
    return None

def _clean_vntl_text(text):
    if not isinstance(text, str):
        return ""
    cleaned = text
    cleaned = cleaned.replace("%name", "<NAME>")
    cleaned = cleaned.replace("%nick", "<NICK>")
    cleaned = cleaned.replace("%fp", "<FP>")
    cleaned = re.sub(r"<\|[^>]+\|>", "", cleaned)
    return cleaned.strip()

def _ignore_segments(ignore_loss):
    if not isinstance(ignore_loss, list):
        return set()
    segs = set()
    for value in ignore_loss:
        if isinstance(value, (int, float)) and int(value) >= 0:
            segs.add(int(value))
    return segs

def _is_nsfw_text(text):
    if not isinstance(text, str):
        return False
    lowered = text.lower()
    keyword_hits = [
        "nsfw",
        "adult",
        "r18",
        "r-18",
        "成人向け",
        "ero",
        "ecchi",
        "hentai",
        "おち●",
    ]
    if any(k in lowered for k in keyword_hits):
        return True
    # Censored-token pattern often used in explicit VN lines.
    if re.search(r"[ぁ-んァ-ン一-龥a-zA-Z]+●", text):
        return True
    return False

def _is_nsfw_row(row):
    if not isinstance(row, dict):
        return False
    text = row.get("text", "")
    if _is_nsfw_text(text):
        return True
    convos = row.get("conversations", None)
    if isinstance(convos, list):
        for turn in convos:
            if not isinstance(turn, dict):
                continue
            content = turn.get("content", None)
            if isinstance(content, str) and _is_nsfw_text(content):
                return True
            if isinstance(content, list):
                for chunk in content:
                    if isinstance(chunk, dict) and _is_nsfw_text(chunk.get("text", "")):
                        return True
    return False

def _extract_vntl_pair(text, ignore_loss=None):
    if not isinstance(text, str):
        return None
    if "<<JAPANESE>>" not in text or "<<ENGLISH>>" not in text:
        return None

    pair_matches = list(re.finditer(
        r"<<JAPANESE>>(.*?)<<ENGLISH>>(.*?)(?=<<JAPANESE>>|$)",
        text,
        flags=re.DOTALL,
    ))
    if not pair_matches:
        return None

    ignored_segment_ids = _ignore_segments(ignore_loss)
    ja_segments = []
    en_segments = []
    for seg_idx, match in enumerate(pair_matches):
        if seg_idx in ignored_segment_ids:
            # ignore_loss marks segments we do not want to supervise in loss.
            continue
        ja_raw = match.group(1)
        en_raw = match.group(2)
        ja_text = _clean_vntl_text(ja_raw)
        en_text = _clean_vntl_text(en_raw)
        if _is_non_empty_text(ja_text) and _is_non_empty_text(en_text):
            ja_segments.append(ja_text)
            en_segments.append(en_text)

    if not ja_segments or not en_segments:
        return None

    user_prompt = (
        "Translate the following Japanese visual-novel segment into natural English. "
        "Preserve speaker tags, tone, and dialogue flow.\n\n"
        f"Japanese:\n{'\n\n'.join(ja_segments)}"
    )
    assistant_text = "\n\n".join(en_segments)
    return user_prompt, assistant_text

def _render_chat_from_pair(user_text, assistant_text):
    convo = [
        {"role": "user", "content": [{"type": "text", "text": user_text.strip()}]},
        {"role": "assistant", "content": [{"type": "text", "text": assistant_text.strip()}]},
    ]
    try:
        return tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix('<bos>')
    except Exception:
        # Last-resort fallback if chat templating fails on this row.
        return f"{user_text.strip()}\n{assistant_text.strip()}"

def _row_to_text_from_batch(batch, idx):
    convo = _get_row_value(batch, "conversations", idx)
    if isinstance(convo, list) and len(convo) > 0:
        try:
            return tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            ).removeprefix('<bos>')
        except Exception:
            pass

    raw_text = _get_row_value(batch, "text", idx)
    raw_ignore_loss = _get_row_value(batch, "ignore_loss", idx)
    vntl_pair = _extract_vntl_pair(raw_text, raw_ignore_loss)
    if vntl_pair is not None:
        return _render_chat_from_pair(vntl_pair[0], vntl_pair[1])

    if _is_non_empty_text(raw_text):
        return raw_text.removeprefix('<bos>')

    candidate_pairs = [
        ("prompt", "response"),
        ("instruction", "output"),
        ("question", "answer"),
        ("input", "output"),
        ("src", "tgt"),
    ]
    for user_key, assistant_key in candidate_pairs:
        user_text = _get_row_value(batch, user_key, idx)
        assistant_text = _get_row_value(batch, assistant_key, idx)
        if _is_non_empty_text(user_text) and _is_non_empty_text(assistant_text):
            return _render_chat_from_pair(user_text, assistant_text)

    return ""

def _is_usable_row(row):
    convos = row.get("conversations", None)
    if isinstance(convos, list) and len(convos) > 0:
        return True

    text = row.get("text", None)
    if _is_non_empty_text(text):
        if _extract_vntl_pair(text, row.get("ignore_loss", None)) is not None:
            return True
        return True

    candidate_pairs = [
        ("prompt", "response"),
        ("instruction", "output"),
        ("question", "answer"),
        ("input", "output"),
        ("src", "tgt"),
    ]
    for user_key, assistant_key in candidate_pairs:
        if _is_non_empty_text(row.get(user_key, None)) and _is_non_empty_text(row.get(assistant_key, None)):
            return True

    return False

def formatting_prompts_func(examples):
    # Batched mapper that supports multiple schema variants.
    n_rows = 0
    for values in examples.values():
        if isinstance(values, list):
            n_rows = len(values)
            break

    texts = [_row_to_text_from_batch(examples, i) for i in range(n_rows)]
    return {"text": texts}

Tokenizer initialized for Gemma-4 chat formatting.


In [8]:
# Tunable NSFW filtering with keyword scoring.
def _nsfw_score(text):
    if not isinstance(text, str):
        return 0
    lowered = text.lower()
    score = 0

    # Strong immediate indicators.
    hard_markers = [
        "nsfw", "hentai", "r18", "r-18", "adult only", "成人向け",
    ]
    if any(k in lowered for k in hard_markers):
        return 100

    explicit_en_terms = [
        "blowjob", "fellatio", "creampie", "cumshot", "ejaculate", "orgasm",
        "penetrat", "thrust", "womb", "inside me", "deeper", "so deep",
        "penis", "vagina", "clitoris", "masturbat", "semen", "erection",
    ]
    explicit_jp_terms = [
        "フェラ", "射精", "中出し", "ちんこ", "まんこ", "セックス",
        "挿入", "子宮", "陰茎", "膣", "絶頂", "愛液", "乳首",
        "おち●", "ち●ん", "ま●こ", "ちん●", "まん●",
    ]
    suggestive_terms = [
        "ero", "ecchi", "エロ", "淫", "えっち", "moan", "panting",
    ]

    for t in explicit_en_terms:
        if t in lowered:
            score += 2
    for t in explicit_jp_terms:
        if t in text:
            score += 2
    for t in suggestive_terms:
        if t in lowered or t in text:
            score += 1

    # Explicit censored-token pattern often used in VN adult lines.
    if re.search(r"(?:おち|ちん|まん|ペニ|ち)[^\n]{0,4}●", text):
        score += 3

    # Moaning-heavy orthography in JP lines (mild signal).
    if re.search(r"[ぁぅぉァゥォ]{2,}.*[ッっ]{1,}", text):
        score += 1

    # Combination signal often found in explicit translated lines.
    if ("womb" in lowered or "子宮" in text) and ("deep" in lowered or "inside" in lowered or "挿入" in text):
        score += 2

    return score

def _is_nsfw_text_tuned(text, level="explicit_only"):
    score = _nsfw_score(text)
    if level == "minimal":
        return score >= 5
    if level == "explicit_only":
        return score >= 2
    if level == "moderate":
        return score >= 1
    return score >= 2

def _is_nsfw_row_tuned(row, level="explicit_only"):
    if not isinstance(row, dict):
        return False

    text = row.get("text", "")
    if _is_nsfw_text_tuned(text, level=level):
        return True

    convos = row.get("conversations", None)
    if isinstance(convos, list):
        for turn in convos:
            if not isinstance(turn, dict):
                continue
            content = turn.get("content", None)
            if isinstance(content, str) and _is_nsfw_text_tuned(content, level=level):
                return True
            if isinstance(content, list):
                for chunk in content:
                    if isinstance(chunk, dict) and _is_nsfw_text_tuned(chunk.get("text", ""), level=level):
                        return True
    return False

#### Filter Usable Rows

In [9]:
# Keep source datasets stable so this section is safe to rerun.
stage1_source = shisa_stage1_raw
stage2_source = stage2_raw
benchmark_source = benchmark_raw

# Keep rows that can be converted to training text.
stage1_filtered = stage1_source.filter(_is_usable_row)
stage2_filtered = stage2_source.filter(_is_usable_row)
benchmark_filtered = benchmark_source.filter(_is_usable_row)

if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_STAGE2:
    stage2_before = len(stage2_filtered)
    stage2_filtered = stage2_filtered.filter(
        lambda row: not _is_nsfw_row_tuned(row, NSFW_FILTER_LEVEL_STAGE2),
        load_from_cache_file=False,
    )
    print(
        f"NSFW filter (stage2, level={NSFW_FILTER_LEVEL_STAGE2}): "
        f"removed {stage2_before - len(stage2_filtered)} rows"
    )

if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_BENCHMARK:
    benchmark_before = len(benchmark_filtered)
    benchmark_filtered = benchmark_filtered.filter(
        lambda row: not _is_nsfw_row_tuned(row, NSFW_FILTER_LEVEL_BENCHMARK),
        load_from_cache_file=False,
    )
    print(
        f"NSFW filter (benchmark, level={NSFW_FILTER_LEVEL_BENCHMARK}): "
        f"removed {benchmark_before - len(benchmark_filtered)} rows"
    )

Filter: 100%|██████████| 13017/13017 [00:00<?, ? examples/s]

Filter: 26034 examples [00:01, 12658.54 examples/s]         


NSFW filter (stage2, level=explicit_only): removed 2396 rows


Filter: 256 examples [00:00, 6817.58 examples/s]        

NSFW filter (benchmark, level=minimal): removed 54 rows


#### Build Formatted Datasets

In [10]:
# Stage 1 formatted dataset
dataset = stage1_filtered.map(
    formatting_prompts_func,
    batched=True,
    keep_in_memory=True,
 )
dataset = dataset.filter(lambda x: _is_non_empty_text(x["text"]))

# Stage 2 formatted dataset
stage2_dataset = stage2_filtered.map(
    formatting_prompts_func,
    batched=True,
    keep_in_memory=True,
 )
stage2_dataset = stage2_dataset.filter(lambda x: _is_non_empty_text(x["text"]))

# Held-out benchmark dataset (never used for training)
benchmark_dataset = benchmark_filtered.map(
    formatting_prompts_func,
    batched=True,
    keep_in_memory=True,
 )
benchmark_dataset = benchmark_dataset.filter(lambda x: _is_non_empty_text(x["text"]))

# Safety net: run NSFW pass again on formatted text to catch residuals.
if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_STAGE2:
    stage2_before_post = len(stage2_dataset)
    stage2_dataset = stage2_dataset.filter(
        lambda row: not _is_nsfw_text_tuned(row.get("text", ""), level=NSFW_FILTER_LEVEL_STAGE2),
        load_from_cache_file=False,
    )
    print(
        f"Post-format NSFW filter (stage2, level={NSFW_FILTER_LEVEL_STAGE2}): "
        f"removed {stage2_before_post - len(stage2_dataset)} rows"
    )

if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_BENCHMARK:
    benchmark_before_post = len(benchmark_dataset)
    benchmark_dataset = benchmark_dataset.filter(
        lambda row: not _is_nsfw_text_tuned(row.get("text", ""), level=NSFW_FILTER_LEVEL_BENCHMARK),
        load_from_cache_file=False,
    )
    print(
        f"Post-format NSFW filter (benchmark, level={NSFW_FILTER_LEVEL_BENCHMARK}): "
        f"removed {benchmark_before_post - len(benchmark_dataset)} rows"
    )

Filter: 100%|██████████| 10621/10621 [00:00<00:00, 17387.49 examples/s]


Post-format NSFW filter (stage2, level=explicit_only): removed 0 rows


Filter: 100%|██████████| 74/74 [00:00<00:00, 14624.63 examples/s]

Post-format NSFW filter (benchmark, level=minimal): removed 0 rows


#### Dataset Summary

In [11]:
print(
    f"Formatted Stage1 rows: {len(dataset)} | "
    f"Formatted Stage2 rows: {len(stage2_dataset)} | "
    f"Formatted benchmark rows: {len(benchmark_dataset)}"
 )

Formatted Stage1 rows: 12000 | Formatted Stage2 rows: 10621 | Formatted benchmark rows: 74


## Dataset Diagnostics
Quick checks for schema quality, usable-row ratios, and token-length distribution before training.

In [12]:
from collections import Counter
from statistics import mean

def _safe_columns(ds):
    try:
        return list(ds.column_names)
    except Exception:
        return []

def _row_type_counters(ds, sample_rows=2000):
    n = min(len(ds), sample_rows)
    counts = Counter()
    for i in range(n):
        row = ds[i]
        if isinstance(row.get("conversations", None), list) and len(row["conversations"]) > 0:
            counts["has_conversations"] += 1
        if isinstance(row.get("text", None), str) and row["text"].strip():
            counts["has_text"] += 1
        for user_key, assistant_key in [("prompt", "response"), ("instruction", "output"), ("question", "answer"), ("input", "output"), ("src", "tgt")]:
            user_text = row.get(user_key, None)
            assistant_text = row.get(assistant_key, None)
            if isinstance(user_text, str) and user_text.strip() and isinstance(assistant_text, str) and assistant_text.strip():
                counts["has_text_pair"] += 1
                break
    return n, counts

datasets_to_check = {
    "stage1_raw": shisa_stage1_raw,
    "stage2_vntl_raw": vntl_stage2_train_raw,
    "stage2_replay_raw": shisa_replay_raw,
    "benchmark_raw": benchmark_raw,
    "stage1_formatted": dataset,
    "stage2_formatted": stage2_dataset,
    "benchmark_formatted": benchmark_dataset,
}

for name, ds in datasets_to_check.items():
    cols = _safe_columns(ds)
    n, counts = _row_type_counters(ds, sample_rows=1500)
    print(f"\n{name}")
    print(f"- rows: {len(ds)}")
    print(f"- columns: {cols}")
    if n > 0:
        print(f"- sample_rows_checked: {n}")
        print(f"- has_conversations: {counts.get('has_conversations', 0)} ({counts.get('has_conversations', 0) / n:.1%})")
        print(f"- has_text: {counts.get('has_text', 0)} ({counts.get('has_text', 0) / n:.1%})")
        print(f"- has_text_pair: {counts.get('has_text_pair', 0)} ({counts.get('has_text_pair', 0) / n:.1%})")


stage1_raw
- rows: 12000
- columns: ['id', 'source_model', 'conversations']
- sample_rows_checked: 1500
- has_conversations: 1500 (100.0%)
- has_text: 0 (0.0%)
- has_text_pair: 0 (0.0%)

stage2_vntl_raw
- rows: 10017
- columns: ['text', 'ignore_loss']
- sample_rows_checked: 1500
- has_conversations: 0 (0.0%)
- has_text: 1500 (100.0%)
- has_text_pair: 0 (0.0%)

stage2_replay_raw
- rows: 3000
- columns: ['id', 'source_model', 'conversations']
- sample_rows_checked: 1500
- has_conversations: 1500 (100.0%)
- has_text: 0 (0.0%)
- has_text_pair: 0 (0.0%)

benchmark_raw
- rows: 128
- columns: ['text', 'ignore_loss']
- sample_rows_checked: 128
- has_conversations: 0 (0.0%)
- has_text: 128 (100.0%)
- has_text_pair: 0 (0.0%)

stage1_formatted
- rows: 12000
- columns: ['id', 'source_model', 'conversations', 'text']
- sample_rows_checked: 1500
- has_conversations: 1500 (100.0%)
- has_text: 1500 (100.0%)
- has_text_pair: 0 (0.0%)

stage2_formatted
- rows: 10621
- columns: ['text', 'ignore_loss', '

In [13]:
import numpy as np

def _token_length_stats(ds, text_field="text", sample_rows=2000):
    lengths = []
    n = min(len(ds), sample_rows)
    for i in range(n):
        text = ds[i].get(text_field, "")
        if isinstance(text, str) and text.strip():
            token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            lengths.append(len(token_ids))
    if not lengths:
        return {"count": 0}
    return {
        "count": len(lengths),
        "mean": float(np.mean(lengths)),
        "p50": float(np.percentile(lengths, 50)),
        "p90": float(np.percentile(lengths, 90)),
        "p95": float(np.percentile(lengths, 95)),
        "p99": float(np.percentile(lengths, 99)),
        "max": int(np.max(lengths)),
    }

for name, ds in [("stage1_formatted", dataset), ("stage2_formatted", stage2_dataset), ("benchmark_formatted", benchmark_dataset)]:
    stats = _token_length_stats(ds, sample_rows=1500)
    print(f"\n{name} token stats")
    for k, v in stats.items():
        print(f"- {k}: {v}")

preview_idx = 0 if len(dataset) == 0 else min(3, len(dataset) - 1)
print("\nStage1 sample text preview:\n")
print(dataset[preview_idx]["text"][:1200])


stage1_formatted token stats
- count: 1500
- mean: 424.026
- p50: 262.5
- p90: 863.3000000000004
- p95: 1202.4499999999996
- p99: 2729.5899999999992
- max: 3663

stage2_formatted token stats
- count: 1500
- mean: 486.552
- p50: 507.0
- p90: 604.0
- p95: 650.0
- p99: 1052.1899999999998
- max: 3681

benchmark_formatted token stats
- count: 74
- mean: 543.5135135135135
- p50: 550.5
- p90: 636.0
- p95: 651.7499999999999
- p99: 675.0
- max: 675

Stage1 sample text preview:

<|turn>user
ある男性に子供は2人います。少なくとも一人は男の子であることはわかっています。二人の子供が両方とも男の子である確率は？<turn|>
<|turn>model
この問題は、条件付き確率と事前確率を考慮する問題です。まず、基本的な事前確率を考えてみましょう。

子供の性別は、男女が等しい確率で生まれるとします（実際には男の方が少し多く生まれる傾向がありますが、ここでは50%の確率で考えることにします）。その場合、ある2人の子供の組み合わせは以下の4つが考えられます。

1. 男の子と男の子（BB）
2. 男の子と女の子（BG）
3. 女の子と男の子（GB）
4. 女の子と女の子（GG）

これらの組み合わせは等しい確率（1/4）で発生すると考えられます。

次に、条件付き確率に注目します。問題では「少なくとも一人は男の子である」という情報が与えられています。これは、次の3つのケースを意味します。

1. BB（両方とも男の子）
2. BG（一人は男の子で、もう一人は女の子）
3. GB（一人は女の子で、もう一人は男の子）

GG（両方とも女の子）は除外されます。上記の3ケースの確率を合計すると、(1/4 + 1/

In [14]:
# Compact VNTL inspection for postprocess decisions
from collections import Counter

def _compact_preview(ds, name, indices):
    print(f"\n{name}")
    for idx in indices:
        row = ds[idx]
        text = row.get("text", "")
        ignore = row.get("ignore_loss", None)
        head = text[:220].replace("\n", " ") if isinstance(text, str) else ""
        print(f"- row={idx} chars={len(text) if isinstance(text, str) else 0} ignore_loss_type={type(ignore).__name__} head={head}")

def _analyze(ds, sample_rows=1200):
    n = min(len(ds), sample_rows)
    counts = Counter()
    for i in range(n):
        text = ds[i].get("text", "")
        ignore = ds[i].get("ignore_loss", None)
        if not isinstance(text, str) or not text.strip():
            counts["empty_or_non_string"] += 1
            continue
        if ignore is not None:
            counts["has_ignore_loss"] += 1
        if "<<JAPANESE>>" in text and "<<ENGLISH>>" in text:
            counts["has_ja_en_markers"] += 1
        if "<<METADATA>>" in text:
            counts["has_metadata_block"] += 1
        if "%name" in text or "%nick" in text or "%fp" in text:
            counts["has_placeholders"] += 1
        if len(text.strip()) < 80:
            counts["very_short"] += 1
    print(f"sampled={n} stats={dict(counts)}")

_compact_preview(vntl_stage2_train_raw, "vntl_stage2_train_raw", [0, 1])
_analyze(vntl_stage2_train_raw)

_compact_preview(vntl_benchmark_raw, "vntl_benchmark_raw", [0, 1])
_analyze(vntl_benchmark_raw)


vntl_stage2_train_raw
- row=0 chars=2055 ignore_loss_type=list head=<<METADATA>> [character] Name: Chocola (ショコラ) | Gender: Female [character] Name: Minaduki Kashou (水無月 嘉祥) | Gender: Male | Aliases: Master (ご主人さま) <<START>> <<JAPANESE>> [嘉祥]: 「なっ……！？」 <<ENGLISH>> [Kashou]: 「Huh...?!」<|e
- row=1 chars=2223 ignore_loss_type=list head=<<METADATA>> [character] Name: Chocola (ショコラ) | Gender: Female [character] Name: Minaduki Kashou (水無月 嘉祥) | Gender: Male | Aliases: Master (ご主人さま) [character] Name: Vanilla (バニラ) | Gender: Female <<START>> <<JAPANESE>> [
sampled=1200 stats={'has_ignore_loss': 1200, 'has_ja_en_markers': 1200, 'has_metadata_block': 1200}

vntl_benchmark_raw
- row=0 chars=1831 ignore_loss_type=list head=<<METADATA>> [character] Name: %name (%name) | Gender: Male | Aliases: %nick (%nick), %fp (%fp) [character] Name: Maisaka Mai (舞阪 茉衣) | Gender: Female <<START>> <<JAPANESE>> [茉衣]: 「おち●ちんの皮を唇でぇ、ちゅぱっ……引っ張りながら横に 　動かして、はむっ、あ
- row=1 chars=2216 ignore_loss_type=list head=<<METADATA

In [15]:
# ignore_loss structure probe on multiple rows
for idx in [0, 1, 2, 3, 10]:
    row = vntl_stage2_train_raw[idx]
    ig = row.get("ignore_loss", None)
    text = row.get("text", "")
    print(f"row={idx} text_len={len(text) if isinstance(text, str) else -1} ignore_type={type(ig).__name__} ignore_len={len(ig) if isinstance(ig, list) else -1}")
    if isinstance(ig, list):
        print("  head:", ig[:20])

row=0 text_len=2055 ignore_type=list ignore_len=2
  head: [8, 12]
row=1 text_len=2223 ignore_type=list ignore_len=3
  head: [8, 9, 10]
row=2 text_len=2299 ignore_type=list ignore_len=1
  head: [6]
row=3 text_len=2214 ignore_type=list ignore_len=0
  head: []
row=10 text_len=2278 ignore_type=list ignore_len=3
  head: [12, 5, 6]


In [16]:
# ignore_loss masking coverage stats
rows_with_ignore = 0
total_ignore_segments = 0
rows_with_markers = 0

for i in range(len(vntl_stage2_train_raw)):
    row = vntl_stage2_train_raw[i]
    txt = row.get("text", "")
    ig = row.get("ignore_loss", None)
    if isinstance(txt, str) and "<<JAPANESE>>" in txt and "<<ENGLISH>>" in txt:
        rows_with_markers += 1
    if isinstance(ig, list) and len(ig) > 0:
        rows_with_ignore += 1
        total_ignore_segments += sum(1 for x in ig if isinstance(x, (int, float)) and int(x) >= 0)

avg_ignored = (total_ignore_segments / rows_with_ignore) if rows_with_ignore else 0.0
print(f"rows_with_ja_en_markers={rows_with_markers}")
print(f"rows_with_ignore_loss_segments={rows_with_ignore}")
print(f"total_ignore_loss_segments={total_ignore_segments}")
print(f"avg_ignored_segments_per_row={avg_ignored:.2f}")

rows_with_ja_en_markers=10017
rows_with_ignore_loss_segments=9221
total_ignore_loss_segments=20248
avg_ignored_segments_per_row=2.20


In [17]:
print("Stage2 formatted sample:\n")
print(stage2_dataset[0]["text"][:900])

print("\nBenchmark formatted sample:\n")
print(benchmark_dataset[0]["text"][:900])

Stage2 formatted sample:

<|turn>user
以下はタスクを説明する指示と、追加の背景情報を提供する入力の組み合わせです。要求を適切に満たす回答を書いてください。

### 指示：
質問に対する回答を文章から一言で抽出してください。回答は名詞で答えてください。 それ以外には何も含めないことを厳守してください。

### 入力：
文章：北アメリカ [SEP] 北アメリカ地域では、最大の都市は上記のとおりメキシコの首都メキシコシティであり、メキシコの人口や富はここに集中している。しかしメキシコ国内ではこのほかにも、グアダラハラとモンテレイ、それにプエブラの3つの都市が人口100万人を超えている。この4つの都市はいずれも人口の集中するメキシコの中央高原に位置している。それに対し、海岸部にはカリブ海岸のベラクルスや太平洋岸のアカプルコなど重要な港湾都市は点在するものの、100万人を超えるような大都市は存在しない。
質問：メキシコの首都は

### 回答：<turn|>
<|turn>model
メキシコシティ<turn|>


Benchmark formatted sample:

<|turn>user
Translate the following Japanese visual-novel segment into natural English. Preserve speaker tags, tone, and dialogue flow.

Japanese:
[茉衣]: 「ふぁぁァァァッ　お豆がっ、クリちゃんがッ、あッ…… 　いじいじされて」

[茉衣]: 「ふぁッ、あァァッ……ヒャ、あぁぁッ……すごい、膣内で 　どんどん膨れ上がってッッ！」

茉衣のリアクションが、いつもより激しいように感じる。

[茉衣]: 「冷たいはずなのに、どんどん熱くなっていっちゃいますね 　これっ」

[<NAME>]: 「でもっ、風邪ひかないように後でお風呂も一緒に入ろうな」

[茉衣]: 「それ、すっごい楽しみですねっ」

[茉衣]: 「はぁっ、あぐぅっ！　はっ、あァッ……　ンッ、 　あんアンッ」

[<NAME>]: 「そろそろ出るぞッ！」

あまりに具合がよすぎて、こっちの方がもたなくなってきた。

[茉衣]: 「ひゃっ……　あっ、たくさん、ん

In [18]:
# Residual NSFW audit on formatted datasets
def _audit_residual_nsfw(ds, level, name, max_preview=3):
    flagged = []
    for i in range(len(ds)):
        text = ds[i].get("text", "")
        if _is_nsfw_text_tuned(text, level=level):
            flagged.append(i)
    print(f"{name}: flagged {len(flagged)} / {len(ds)}")
    for j, idx in enumerate(flagged[:max_preview]):
        snippet = ds[idx].get("text", "")[:260].replace("\n", " ")
        print(f"- {name} flagged idx={idx} snippet={snippet}")

_audit_residual_nsfw(stage2_dataset, NSFW_FILTER_LEVEL_STAGE2, "stage2_dataset")
_audit_residual_nsfw(benchmark_dataset, NSFW_FILTER_LEVEL_BENCHMARK, "benchmark_dataset")

stage2_dataset: flagged 0 / 10621
benchmark_dataset: flagged 0 / 74


## Model Setup
Load Gemma 4 31B in 4-bit and keep LoRA detached for pre-train benchmarking.

In [19]:
import gc

if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

visible_gpus = torch.cuda.device_count()
print(f"Visible GPUs: {visible_gpus}")
for i in range(visible_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"- cuda:{i} {props.name} | VRAM {props.total_memory / 1024**3:.2f} GiB")

extra_load_kwargs = {}
if USE_MULTI_GPU_MODEL_SHARDING and visible_gpus > 1:
    max_memory = {}
    for i in range(visible_gpus):
        total_gib = int(torch.cuda.get_device_properties(i).total_memory / 1024**3)
        reserve = GPU0_RESERVED_GIB if i == 0 else OTHER_GPU_RESERVED_GIB
        alloc = max(8, total_gib - reserve)
        max_memory[i] = f"{alloc}GiB"
    max_memory["cpu"] = "128GiB"

    # Unsloth docs path: device_map="balanced" for model splitting across GPUs.
    extra_load_kwargs = {
        "device_map": MULTI_GPU_DEVICE_MAP,
        "max_memory": max_memory,
    }
    print(f"Attempting model split with device_map={MULTI_GPU_DEVICE_MAP}")
    print(f"max_memory={max_memory}")
else:
    print("Using single-device model placement.")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        **extra_load_kwargs,
     )
except ValueError as e:
    if "max_memory" in extra_load_kwargs:
        print(f"max_memory rejected ({e}). Retrying with device_map only.")
        fallback_kwargs = {k: v for k, v in extra_load_kwargs.items() if k != "max_memory"}
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
            **fallback_kwargs,
         )
    else:
        raise
except TypeError as e:
    print(f"Multi-GPU kwargs not accepted in this environment ({e}). Falling back to default load.")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
     )

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
 )

BASE_BENCHMARK = None
TRAINED_BENCHMARK = None
LORA_ATTACHED = False

param_devices = sorted({str(p.device) for p in model.parameters()})
print(f"Model parameter devices: {param_devices}")
print(f"Loaded base model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

Visible GPUs: 2
- cuda:0 NVIDIA GeForce RTX 5090 | VRAM 31.84 GiB
- cuda:1 NVIDIA GeForce RTX 3090 | VRAM 24.00 GiB
Attempting model split with device_map=balanced
max_memory={0: '26GiB', 1: '20GiB', 'cpu': '128GiB'}
==((====))==  Unsloth 2026.4.1: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 2. Max memory: 31.842 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 1188/1188 [00:20<00:00, 57.27it/s] 


Model parameter devices: ['cuda:0', 'cuda:1']
Loaded base model: unsloth/gemma-4-31B-it
Max sequence length: 8192


In [20]:
import torch

def ensure_lora_model():
    global model, tokenizer, LORA_ATTACHED

    model_missing = ("model" not in globals()) or (model is None)
    if model_missing:
        print("Base model is missing in memory. Reloading before LoRA attach...")
        reload_kwargs = {}
        visible_gpus = torch.cuda.device_count()
        if USE_MULTI_GPU_MODEL_SHARDING and visible_gpus > 1:
            reload_kwargs["device_map"] = MULTI_GPU_DEVICE_MAP

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
            **reload_kwargs,
        )
        tokenizer = get_chat_template(
            tokenizer,
            chat_template="gemma-4",
        )
        LORA_ATTACHED = False
        print("Base model reloaded.")

    if LORA_ATTACHED:
        print("LoRA adapters are already attached.")
        return

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    LORA_ATTACHED = True
    print(f"LoRA attached with r={LORA_R}, alpha={LORA_ALPHA}")

print("LoRA helper ready. It will auto-reload the base model if missing.")

LoRA helper ready. It will auto-reload the base model if missing.


## Benchmark Helpers
Evaluate baseline and post-training quality with held-out benchmark rows.

In [21]:
import math
import statistics
import time

def compute_eval_loss_direct(model_obj, tokenizer_obj, eval_ds, batch_size=1, max_length=MAX_SEQ_LENGTH):
    texts = [x["text"] for x in eval_ds if isinstance(x.get("text", None), str) and x["text"].strip()]
    if len(texts) == 0:
        return float("nan")

    total_neg_log_likelihood = 0.0
    total_tokens = 0
    model_obj.eval()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        # Gemma processor requires explicit text=... to avoid treating input as image/video args.
        enc = tokenizer_obj(
            text=batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}

        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        valid_tokens = int((labels != -100).sum().item())
        if valid_tokens == 0:
            continue

        with torch.inference_mode():
            outputs = model_obj(**enc, labels=labels)
            loss_value = float(outputs.loss.item())

        total_neg_log_likelihood += loss_value * valid_tokens
        total_tokens += valid_tokens

    if total_tokens == 0:
        return float("nan")
    return total_neg_log_likelihood / total_tokens

BENCHMARK_PROMPTS = [
    "Translate this natural VN line to smooth English while preserving tone: ほんとに、今日はありがと。",
    "Rewrite this EN line with softer VN tone: I guess I can stay for a bit longer.",
    "Continue this dialogue naturally in VN style: [A] Did you really come all this way? [B]",
]

def run_generation_benchmark(model_obj, tokenizer_obj, prompts, max_new_tokens=80):
    def measure_generation(prompt):
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        inputs = tokenizer_obj.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to("cuda")

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            outputs = model_obj.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        generated_tokens = int(outputs.shape[-1] - inputs["input_ids"].shape[-1])
        tok_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
        return elapsed, generated_tokens, tok_per_sec

    _ = measure_generation("Warmup run for generation benchmark.")
    results = [measure_generation(p) for p in prompts]
    latencies = [r[0] for r in results]
    tokens = [r[1] for r in results]
    tps = [r[2] for r in results]
    return {
        "avg_latency_sec": statistics.mean(latencies),
        "avg_generated_tokens": statistics.mean(tokens),
        "avg_tokens_per_sec": statistics.mean(tps),
        "median_tokens_per_sec": statistics.median(tps),
    }

def run_model_benchmark(model_obj, tokenizer_obj, eval_ds, label):
    eval_loss = compute_eval_loss_direct(model_obj, tokenizer_obj, eval_ds)
    perplexity = math.exp(eval_loss) if isinstance(eval_loss, (float, int)) and math.isfinite(eval_loss) and eval_loss < 20 else float("inf")
    gen_metrics = run_generation_benchmark(model_obj, tokenizer_obj, BENCHMARK_PROMPTS, max_new_tokens=80)
    metrics = {
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        **gen_metrics,
    }

    print(f"\n{label}")
    print(f"- eval_loss: {metrics['eval_loss']:.4f}")
    print(f"- perplexity: {metrics['perplexity']:.4f}")
    print(f"- avg_latency_sec: {metrics['avg_latency_sec']:.3f}")
    print(f"- avg_generated_tokens: {metrics['avg_generated_tokens']:.1f}")
    print(f"- avg_tokens_per_sec: {metrics['avg_tokens_per_sec']:.2f}")
    print(f"- median_tokens_per_sec: {metrics['median_tokens_per_sec']:.2f}")
    return metrics

## Base Benchmark
Run this before LoRA attachment/training for a clean baseline comparison.

In [22]:
if len(benchmark_dataset) == 0:
    raise RuntimeError(
        "benchmark_dataset is empty. Re-run dataset build cells before benchmarking."
    )

if LORA_ATTACHED:
    raise RuntimeError(
        "LoRA is already attached. Restart and run in order to capture true base benchmark."
    )

BASE_BENCHMARK = run_model_benchmark(
    model,
    tokenizer,
    benchmark_dataset,
    "Base model benchmark (pre-LoRA)",
)


Base model benchmark (pre-LoRA)
- eval_loss: 4.6210
- perplexity: 101.5921
- avg_latency_sec: 13.913
- avg_generated_tokens: 80.0
- avg_tokens_per_sec: 5.75
- median_tokens_per_sec: 5.77


## Two-Stage Training
Stage 1: Shisa warmup. Stage 2: VNTL specialization + replay.

### Optional: True 2-GPU Distributed Launch
This cell exports datasets and builds an `accelerate launch` command that uses both GPUs via data parallel training.

In [23]:
from pathlib import Path
import gc
import os
import platform
import shlex
import subprocess
import sys

# 1) Export current formatted datasets for distributed script input
DIST_DATA_DIR = Path("outputs/distributed_data")
DIST_DATA_DIR.mkdir(parents=True, exist_ok=True)

stage1_data_path = DIST_DATA_DIR / "stage1_text.parquet"
stage2_data_path = DIST_DATA_DIR / "stage2_text.parquet"
benchmark_data_path = DIST_DATA_DIR / "benchmark_text.parquet"

dataset.select_columns(["text"]).to_parquet(str(stage1_data_path))
stage2_dataset.select_columns(["text"]).to_parquet(str(stage2_data_path))
benchmark_dataset.select_columns(["text"]).to_parquet(str(benchmark_data_path))

print(f"Saved stage1 parquet: {stage1_data_path} rows={len(dataset)}")
print(f"Saved stage2 parquet: {stage2_data_path} rows={len(stage2_dataset)}")
print(f"Saved benchmark parquet: {benchmark_data_path} rows={len(benchmark_dataset)}")

# Resolve distributed script path robustly for either notebook CWD style.
script_candidates = [
    Path("train_distributed.py"),
    Path("unsloth") / "train_distributed.py",
]
script_path = next((p for p in script_candidates if p.exists()), None)
if script_path is None:
    raise FileNotFoundError(
        f"Could not find train_distributed.py. Tried: {[str(p) for p in script_candidates]}"
    )
script_path = script_path.resolve()
print(f"Using distributed script: {script_path}")

# Optional fast sanity run before full distributed training.
DISTRIBUTED_PREFLIGHT = True
pre_stage1_steps = 1 if DISTRIBUTED_PREFLIGHT else STAGE1_MAX_STEPS
pre_stage2_steps = 1 if DISTRIBUTED_PREFLIGHT else STAGE2_MAX_STEPS
stage1_out = Path("outputs/stage1_ddp_smoke" if DISTRIBUTED_PREFLIGHT else "outputs/stage1_ddp").resolve()
stage2_out = Path("outputs/stage2_ddp_smoke" if DISTRIBUTED_PREFLIGHT else "outputs/stage2_ddp").resolve()
final_out = Path("outputs/gemma4_31b_lora_final_ddp_smoke" if DISTRIBUTED_PREFLIGHT else "outputs/gemma4_31b_lora_final_ddp").resolve()
print(f"DISTRIBUTED_PREFLIGHT={DISTRIBUTED_PREFLIGHT} | stage1_steps={pre_stage1_steps} | stage2_steps={pre_stage2_steps}")

common_train_args = [
    str(script_path),
    "--model_name", MODEL_NAME,
    "--stage1_data", str(stage1_data_path.resolve()),
    "--stage2_data", str(stage2_data_path.resolve()),
    "--max_seq_length", str(min(MAX_SEQ_LENGTH, 4096)),
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--stage1_max_steps", str(pre_stage1_steps),
    "--stage2_max_steps", str(pre_stage2_steps),
    "--stage1_lr", str(STAGE1_LR),
    "--stage2_lr", str(STAGE2_LR),
    "--lora_r", str(LORA_R),
    "--lora_alpha", str(LORA_ALPHA),
    "--stage1_output_dir", str(stage1_out),
    "--stage2_output_dir", str(stage2_out),
    "--final_output_dir", str(final_out),
]

# 2) Build launch commands (both are valid per Unsloth docs).
ACCELERATE_ARGS = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--num_processes", "2",
    "--mixed_precision", "bf16",
    *common_train_args,
]

TORCHRUN_ARGS = [
    sys.executable, "-m", "torch.distributed.run",
    "--nproc_per_node", "2",
    *common_train_args,
]

ACCELERATE_CMD = " ".join(shlex.quote(x) for x in ACCELERATE_ARGS)
TORCHRUN_CMD = " ".join(shlex.quote(x) for x in TORCHRUN_ARGS)

print("\nAccelerate launch command:")
print(ACCELERATE_CMD)
print("\nTorchrun launch command:")
print(TORCHRUN_CMD)

IS_WINDOWS = platform.system() == "Windows"
if IS_WINDOWS:
    print("\nDetected native Windows runtime.")
    print("This PyTorch build fails distributed rendezvous with: use_libuv was requested but libuv is unavailable.")
    print("Recommended: run distributed training under WSL2/Linux, or use in-notebook model sharding on Windows.")

def _tail_block(text, n_lines=30, max_line=220):
    lines = text.splitlines()[-n_lines:]
    compact = []
    for line in lines:
        if len(line) > max_line:
            compact.append(line[:max_line] + " ...<truncated>")
        else:
            compact.append(line)
    return "\n".join(compact)

# 3) Optional launch toggle
RUN_DISTRIBUTED_TRAINING = False
DISTRIBUTED_LAUNCHER = "accelerate"  # choices: accelerate, torchrun
ALLOW_WINDOWS_DDP_EXPERIMENT = False

if RUN_DISTRIBUTED_TRAINING:
    if IS_WINDOWS and not ALLOW_WINDOWS_DDP_EXPERIMENT:
        print("Skipping launch because native-Windows DDP is blocked in this environment.")
        print("Set ALLOW_WINDOWS_DDP_EXPERIMENT=True if you still want to try and inspect logs.")
    else:
        # Free notebook-held model VRAM before spawning DDP workers.
        if "model" in globals():
            print("Releasing notebook model from GPU memory before distributed launch...")
            try:
                del model
            except Exception:
                pass
            gc.collect()
            torch.cuda.empty_cache()

        selected_args = ACCELERATE_ARGS if DISTRIBUTED_LAUNCHER == "accelerate" else TORCHRUN_ARGS
        print(f"Launching distributed training with {DISTRIBUTED_LAUNCHER}...")

        launch_env = os.environ.copy()
        launch_env["USE_LIBUV"] = "0"
        print("USE_LIBUV=0 set for distributed launch.")

        completed = subprocess.run(
            selected_args,
            text=True,
            capture_output=True,
            env=launch_env,
        )

        stdout_log = DIST_DATA_DIR / "ddp_last_stdout.log"
        stderr_log = DIST_DATA_DIR / "ddp_last_stderr.log"
        stdout_log.write_text(completed.stdout or "", encoding="utf-8")
        stderr_log.write_text(completed.stderr or "", encoding="utf-8")
        print(f"Saved logs: {stdout_log} and {stderr_log}")

        if completed.returncode != 0:
            print(f"Distributed launch failed with return code {completed.returncode}.")
            print("\n--- subprocess stdout (compact tail) ---")
            print(_tail_block(completed.stdout or ""))
            print("\n--- subprocess stderr (compact tail) ---")
            print(_tail_block(completed.stderr or ""))
            raise RuntimeError("Distributed training launch failed. See compact tail above and saved log files.")

        print("Distributed training completed successfully.")
else:
    print("\nRUN_DISTRIBUTED_TRAINING is False. Set it to True to launch 2-GPU training.")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 999.83ba/s]

Saved stage1 parquet: outputs\distributed_data\stage1_text.parquet rows=12000
Saved stage2 parquet: outputs\distributed_data\stage2_text.parquet rows=10621
Saved benchmark parquet: outputs\distributed_data\benchmark_text.parquet rows=74
Using distributed script: C:\Users\Peter\sub01\unsloth\train_distributed.py
DISTRIBUTED_PREFLIGHT=True | stage1_steps=1 | stage2_steps=1

Accelerate launch command:
'c:\Users\li\miniconda3\envs\unsloth\python.exe' -m accelerate.commands.launch --num_processes 2 --mixed_precision bf16 'C:\Users\Peter\sub01\unsloth\train_distributed.py' --model_name unsloth/gemma-4-31B-it --stage1_data 'C:\Users\Peter\sub01\unsloth\outputs\distributed_data\stage1_text.parquet' --stage2_data 'C:\Users\Peter\sub01\unsloth\outputs\distributed_data\stage2_text.parquet' --max_seq_length 4096 --per_device_train_batch_size 1 --gradient_accumulation_steps 8 --stage1_max_steps 1 --stage2_max_steps 1 --stage1_lr 0.00012 --stage2_lr 7e-05 --lora_r 8 --lora_alpha 16 --stage1_output_d

In [ ]:
import os
import sys
import torch.nn.functional as F

def _install_tensorwrapper_fallback(chunk_tokens=1024):
    # Workaround for Windows+TensorWrapper failures in Unsloth fused CE autograd.
    def _safe_unsloth_fused_ce_loss(
        trainer,
        hidden_states,
        lm_head_weight,
        lm_head_bias,
        labels,
        mask=None,
        n_items=None,
        scaling=None,
        target_gb=None,
        torch_compile=True,
        overwrite=False,
        **kwargs,
    ):
        device = lm_head_weight.device
        hidden_states = hidden_states.to(device=device, dtype=lm_head_weight.dtype)

        # Shift labels to match next-token prediction.
        shifted_states = hidden_states[..., :-1, :].contiguous()
        shifted_labels = labels[..., 1:].contiguous().clone()
        if mask is not None:
            shifted_mask = mask[..., 1:].to(device=shifted_labels.device)
            shifted_labels = shifted_labels.masked_fill(shifted_mask == 0, -100)

        flat_states = shifted_states.view(-1, shifted_states.shape[-1])
        flat_labels = shifted_labels.view(-1).to(device=device)

        logit_scale_multiply = kwargs.get("logit_scale_multiply", 0)
        logit_scale_divide = kwargs.get("logit_scale_divide", 0)
        logit_softcapping = kwargs.get("logit_softcapping", 0)

        total_loss = torch.zeros((), device=device, dtype=torch.float32)
        total_valid = torch.zeros((), device=device, dtype=torch.float32)

        for start in range(0, flat_labels.numel(), chunk_tokens):
            end = min(start + chunk_tokens, flat_labels.numel())
            hs_chunk = flat_states[start:end]
            lb_chunk = flat_labels[start:end]

            logits = F.linear(hs_chunk, lm_head_weight, lm_head_bias).float()
            if logit_scale_multiply not in (None, 0):
                logits = logits * logit_scale_multiply
            if logit_scale_divide not in (None, 0):
                logits = logits / logit_scale_divide
            if logit_softcapping not in (None, 0):
                logits = logits / logit_softcapping
                logits = torch.tanh(logits)
                logits = logits * logit_softcapping

            chunk_loss = F.cross_entropy(
                logits,
                lb_chunk,
                ignore_index=-100,
                reduction="sum",
            )
            total_loss = total_loss + chunk_loss
            total_valid = total_valid + (lb_chunk != -100).sum().to(torch.float32)

        if n_items is not None:
            if torch.is_tensor(n_items):
                denom = n_items.to(device=device, dtype=torch.float32)
            else:
                denom = torch.tensor(float(n_items), device=device, dtype=torch.float32)
        else:
            denom = torch.clamp(total_valid, min=1.0)

        return total_loss / denom

    patched = 0
    for mod_name, mod in list(sys.modules.items()):
        if mod is None:
            continue
        if mod_name.startswith("unsloth_compiled_module_") and hasattr(mod, "unsloth_fused_ce_loss"):
            setattr(mod, "unsloth_fused_ce_loss", _safe_unsloth_fused_ce_loss)
            patched += 1

    try:
        import unsloth_zoo.loss_utils as _loss_utils
        _loss_utils.unsloth_fused_ce_loss = _safe_unsloth_fused_ce_loss
    except Exception:
        pass

    print(f"Installed TensorWrapper fallback CE in {patched} compiled module(s); chunk_tokens={chunk_tokens}")

_fallback_chunk_tokens = int(os.environ.get("UNSLOTH_SAFE_CE_CHUNK_TOKENS", "1024"))
_install_tensorwrapper_fallback(chunk_tokens=max(128, _fallback_chunk_tokens))

if len(dataset) == 0:
    raise RuntimeError("Stage 1 dataset is empty. Check formatting diagnostics above.")
if len(stage2_dataset) == 0:
    raise RuntimeError("Stage 2 dataset is empty. Check formatting diagnostics above.")

ensure_lora_model()

trainer_stage1 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_steps=max(8, STAGE1_MAX_STEPS // 10),
        max_steps=STAGE1_MAX_STEPS,
        learning_rate=STAGE1_LR,
        logging_steps=10,
        save_steps=max(20, STAGE1_MAX_STEPS // 4),
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        seed=3407,
        report_to="none",
    ),
)
trainer_stage1 = train_on_responses_only(
    trainer_stage1,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

print("Starting Stage 1 training...")
trainer_stats_stage1 = trainer_stage1.train()
trainer_stage1.save_model(OUTPUT_DIR)
print(f"Stage 1 adapter saved to: {OUTPUT_DIR}")

trainer_stage2 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=stage2_dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_steps=max(8, STAGE2_MAX_STEPS // 10),
        max_steps=STAGE2_MAX_STEPS,
        learning_rate=STAGE2_LR,
        logging_steps=10,
        save_steps=max(20, STAGE2_MAX_STEPS // 4),
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        seed=3407,
        report_to="none",
    ),
)
trainer_stage2 = train_on_responses_only(
    trainer_stage2,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

print("Starting Stage 2 training...")
trainer_stats = trainer_stage2.train()
trainer_stage2.save_model(STAGE2_OUTPUT_DIR)
print(f"Stage 2 adapter saved to: {STAGE2_OUTPUT_DIR}")

# Keep compatibility for later cells that expect `trainer`
trainer = trainer_stage2

Installed TensorWrapper fallback CE in 2 compiled module(s); chunk_tokens=1024
Unsloth: Making `model.base_model.model.model.language_model` require gradients
LoRA attached with r=8, alpha=16


Filter: 100%|██████████| 12000/12000 [00:04<00:00, 2688.13 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Unsloth: Removed 1 out of 12000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.
Starting Stage 1 training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 11,999 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 66,740,864 of 31,339,827,376 (0.21% trained)


Unsloth: Will smartly offload gradients to save VRAM!


...

Filter: 100%|██████████| 10621/10621 [00:04<00:00, 2186.43 examples/s]


Starting Stage 2 training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 10,621 | Num Epochs = 1 | Total steps = 700
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 66,740,864 of 31,339,827,376 (0.21% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.952864
20,0.999962
30,0.923410
40,0.926829
50,0.841400
60,0.867799
70,0.885662
80,0.956092
90,0.868774
100,0.807879


## Post-Train Benchmark
Evaluate the trained adapter on the same held-out benchmark split used for the base model.

In [ ]:
TRAINED_BENCHMARK = run_model_benchmark(
    model,
    tokenizer,
    benchmark_dataset,
    "Trained model benchmark",
)

if BASE_BENCHMARK is not None:
    print("\nDelta vs base model (trained - base)")
    print(f"- eval_loss_delta: {TRAINED_BENCHMARK['eval_loss'] - BASE_BENCHMARK['eval_loss']:.4f} (lower is better)")
    print(f"- perplexity_delta: {TRAINED_BENCHMARK['perplexity'] - BASE_BENCHMARK['perplexity']:.4f} (lower is better)")
    print(f"- avg_tokens_per_sec_delta: {TRAINED_BENCHMARK['avg_tokens_per_sec'] - BASE_BENCHMARK['avg_tokens_per_sec']:.2f} (higher is better)")
    print(f"- median_tokens_per_sec_delta: {TRAINED_BENCHMARK['median_tokens_per_sec'] - BASE_BENCHMARK['median_tokens_per_sec']:.2f} (higher is better)")
else:
    print("BASE_BENCHMARK is missing, so delta comparison was skipped.")

## Save Adapter
Save the final LoRA adapter for reuse and optional export.

In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Saved final LoRA adapter to: {FINAL_ADAPTER_DIR}")

# Optional GGUF export (set True to run; this can take a while).
EXPORT_GGUF = False
if EXPORT_GGUF:
    model.save_pretrained_gguf(
        "outputs/gemma4_31b_gguf",
        tokenizer,
        quantization_method="Q8_0",
    )
    print("Saved GGUF to outputs/gemma4_31b_gguf")

## Quick Inference Smoke Test
Sanity-check generation with the trained model in chat format.

In [ ]:
from transformers import TextStreamer

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "Write a short bilingual Japanese-English greeting."}],
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens=96,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)